# 01. Setup & Cleaning
In questo notebook viene implementata la pipeline iniziale per il caricamento e la pulizia dei dati relativi al mondo degli anime. I dati 'sanificati' vengono poi salvati nella cartella `data/processed/` per le successive fasi di analisi e visualizzazione.

### Obbiettivi
- Caricamento dei CSV: importazione dei file necessari per l'analisi.
- Ispezione e pulizia dei dati: ricerca di anomalie e rimozione dei duplicati.
- Gestione valori nulli e tipi di dato: trattamento dei dati mancanti e correzione dei formati.
- Salvataggio dei CSV puliti: Esportazione dei dati pronti all'uso nella cartella `data/processed/`.

In [44]:
import pandas as pd
import numpy as np
%load_ext autoreload
%autoreload 2
from functions.utils import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Caricamento Dataset

In questa sezione carichiamo i file CSV originali per poi organizzarli all'interno di un dizionario; questo approccio ci permette di lavorare in modo più efficace durante le fasi di pulizia e ispezione, facilitando la gestione simultanea di tutti i dataset.

In [2]:
characters_df = pd.read_csv("../data/datasets/characters.csv")
character_anime_works_df = pd.read_csv("../data/datasets/character_anime_works.csv")
character_nicknames_df = pd.read_csv("../data/datasets/character_nicknames.csv")
details_df = pd.read_csv("../data/datasets/details.csv")
favs_df = pd.read_csv("../data/datasets/favs.csv")
person_alternate_names_df = pd.read_csv("../data/datasets/person_alternate_names.csv")
person_anime_works_df = pd.read_csv("../data/datasets/person_anime_works.csv")
person_voice_works_df = pd.read_csv("../data/datasets/person_voice_works.csv")
person_details_df = pd.read_csv("../data/datasets/person_details.csv")
profiles_df = pd.read_csv("../data/datasets/profiles.csv")
ratings_df = pd.read_csv("../data/datasets/ratings.csv")
recommendations_df = pd.read_csv("../data/datasets/recommendations.csv")
stats_df = pd.read_csv("../data/datasets/stats.csv")

Stampiamo la dimensione di ciascun dataset per verificare l'esito dell'importazione.

In [28]:
datasets = {
    "Characters": characters_df,
    "Character_anime_works": character_anime_works_df,
    "Character_nicknames": character_nicknames_df,
    "Details": details_df,
    "Favs": favs_df,
    "Person_alternate_names": person_alternate_names_df,
    "Person_anime_works": person_anime_works_df,
    "Person_voice_works": person_voice_works_df,
    "Person_details": person_details_df,
    "Profiles": profiles_df,
    "Ratings": ratings_df,
    "Recommendations": recommendations_df,
    "Stats": stats_df
}

for name, df in datasets.items():
     show_shape(df, name)

Characters: 209963 rows, 7 columns
Character_anime_works: 236816 rows, 4 columns
Character_nicknames: 37080 rows, 2 columns
Details: 28955 rows, 29 columns
Favs: 4178747 rows, 3 columns
Person_alternate_names: 20465 rows, 2 columns
Person_anime_works: 458091 rows, 3 columns
Person_voice_works: 489516 rows, 5 columns
Person_details: 76699 rows, 10 columns
Profiles: 337155 rows, 10 columns
Ratings: 124298357 rows, 6 columns
Recommendations: 105249 rows, 2 columns
Stats: 28955 rows, 27 columns


## 2. Ispezione Dataset

Visualizziamo le prime righe di `Characters`, `Details` e `Profiles` come campioni rappresentativi per capire la struttura generale dei dati.

In [32]:
display(datasets["Characters"].head())
display(datasets["Details"].head())
display(datasets["Profiles"].head())

,character_mal_id,url,name,name_kanji,image,favorites,about
0,280386.0,https://myanimelist.net/character/280386/Envi_...,Envi Mel Champagne,エンヴィ・メル・シャンパーニュ,https://cdn.myanimelist.net/images/characters/...,0.0,NaN
1,280354.0,https://myanimelist.net/character/280354/Eleven,Eleven,イレヴン,https://cdn.myanimelist.net/images/characters/...,0.0,NaN
2,280353.0,https://myanimelist.net/character/280353/Stud,Stud,スタッド,https://cdn.myanimelist.net/images/characters/...,0.0,NaN
3,280352.0,https://myanimelist.net/character/280352/Judge,Judge,ジャッジ,https://cdn.myanimelist.net/images/characters/...,0.0,NaN
4,280339.0,https://myanimelist.net/character/280339/Eiji_...,Eiji Kurokawa,黒川 英治,https://cdn.myanimelist.net/img/sp/icon/apple-...,0.0,NaN


,mal_id,title,title_japanese,url,image_url,type,status,score,scored_by,start_date,...,demographics,source,rating,episodes,season,year,producers,explicit_genres,licensors,streaming
0,59356,-Socket-,-socket-,https://myanimelist.net/anime/59356/-Socket-,https://cdn.myanimelist.net/images/anime/1043/...,Movie,Finished Airing,NaN,<NA>,2010-01-01 00:00:00+00:00,...,[],Original,G - All Ages,1,NaN,<NA>,['Nagoya Zokei University'],[],[],[]
1,56036,......,......,https://myanimelist.net/anime/56036/-,https://cdn.myanimelist.net/images/anime/1057/...,Music,Finished Airing,6.53,503,2023-06-11 00:00:00+00:00,...,[],Original,PG-13 - Teens 13 or older,1,NaN,<NA>,[],[],[],[]
2,2928,.hack//G.U. Returner,.HACK//G.U. RETURNER,https://myanimelist.net/anime/2928/hack__GU_Re...,https://cdn.myanimelist.net/images/anime/1798/...,OVA,Finished Airing,6.65,9745,2007-01-18 00:00:00+00:00,...,[],Game,PG-13 - Teens 13 or older,1,NaN,<NA>,"['Bandai Visual', 'CyberConnect2']",[],[],[]
3,3269,.hack//G.U. Trilogy,.hack//G.U. Trilogy,https://myanimelist.net/anime/3269/hack__GU_Tr...,https://cdn.myanimelist.net/images/anime/1566/...,Movie,Finished Airing,7.06,15373,2007-12-22 00:00:00+00:00,...,[],Game,PG-13 - Teens 13 or older,1,NaN,<NA>,['Bandai Visual'],[],"['Funimation', 'Bandai Entertainment']",[]
4,4469,.hack//G.U. Trilogy: Parody Mode,.hack//G.U. Trilogy,https://myanimelist.net/anime/4469/hack__GU_Tr...,https://cdn.myanimelist.net/images/anime/10/86...,Special,Finished Airing,6.35,4317,2008-03-25 00:00:00+00:00,...,[],Game,PG-13 - Teens 13 or older,1,NaN,<NA>,['Bandai Visual'],[],[],[]


,username,gender,birthday,location,joined,watching,completed,on_hold,dropped,plan_to_watch
0,ishikawas,NaN,NaT,South Korea,NaT,<NA>,<NA>,<NA>,<NA>,<NA>
1,CKK2,NaN,NaT,United States,2018-12-01,3,182,15,0,405
2,--------788,Female,NaT,Mexico,2022-10-04,1,64,0,0,1
3,potatoaris,NaN,NaT,Spain,2018-10-02,5,1,0,0,4
4,Rinrintan,NaN,NaT,Japan,2019-05-12,20,311,40,16,34


## 3. Rimozione dei Duplicati

In questa sezione analizziamo l'integrità dei dati cercando eventuali righe duplicate in tutti i dataset. Sfruttiamo il dizionario creato in precedenza per applicare la funzione `check_duplicates` in modo iterativo; questo ci permette di avere una panoramica immediata dei duplicati presenti e di procedere alla loro rimozione.

In [5]:
for name, df in datasets.items():
    check_duplicates(df, name)
    print()

In 'Characters': 340 original records produced 456 extra copies.

In 'Character_anime_works': No duplicates found.

In 'Character_nicknames': 66 original records produced 102 extra copies.

In 'Details': No duplicates found.

In 'Favs': No duplicates found.

In 'Person_alternate_names': 29 original records produced 39 extra copies.

In 'Person_anime_works': No duplicates found.

In 'Person_voice_works': 537 original records produced 951 extra copies.

In 'Person_details': 3 original records produced 3 extra copies.

In 'Profiles': No duplicates found.

In 'Ratings': No duplicates found.

In 'Recommendations': No duplicates found.

In 'Stats': No duplicates found.



Successivamente, utilizziamo la funzione `clean_duplicates` per rimuoverli definitivamente, garantendo che ogni dataset contenga solo informazioni univoche e che il numero di righe rimosse corrisponda alle copie rilevate prima.

In [6]:
datasets_to_clean = [
    "Characters", 
    "Character_nicknames", 
    "Person_alternate_names", 
    "Person_voice_works", 
    "Person_details"
]

for name in datasets_to_clean:
    if name in datasets:
        old_len = len(datasets[name])
        datasets[name] = clean_duplicates(datasets[name])
        print(f"Dataset {name:<25}: new rows {len(datasets[name])}, before was {old_len} rows, removed {old_len - len(datasets[name])}")

Dataset Characters               : new rows 209507, before was 209963 rows, removed 456
Dataset Character_nicknames      : new rows 36978, before was 37080 rows, removed 102
Dataset Person_alternate_names   : new rows 20426, before was 20465 rows, removed 39
Dataset Person_voice_works       : new rows 488565, before was 489516 rows, removed 951
Dataset Person_details           : new rows 76696, before was 76699 rows, removed 3


## 4. Definizione dei Tipi di Dato (dtypes)

In questa sezione impostiamo il formato corretto per le colonne di ogni dataset.   
Iniziamo visualizzando i tipi di dato di ogni dataset utilizzando l'attributo `.dtypes`.

In [42]:
for name, df in datasets.items():
    print(f"\nDATASET: {name}")
    print(df.dtypes)


DATASET: Characters
character_mal_id     int64
url                 object
name                object
name_kanji          object
image               object
favorites            Int64
about               object
dtype: object

DATASET: Character_anime_works
anime_mal_id         int64
character_mal_id     int64
character_name      object
role                object
dtype: object

DATASET: Character_nicknames
character_mal_id     int64
nickname            object
dtype: object

DATASET: Details
mal_id                           int64
title                           object
title_japanese                  object
url                             object
image_url                       object
type                            object
status                          object
score                          float64
scored_by                        Int64
start_date         datetime64[ns, UTC]
end_date           datetime64[ns, UTC]
synopsis                        object
rank                             Int64

Trasformiamo tutte le colonne contenenti date nel formato datetime, utilizzando la funzione `convert_to_datetime` per la gestione dei valori nulli e delle date non valide.

In [34]:
datasets["Person_details"] = convert_to_datetime(datasets["Person_details"], "birthday")

datasets["Profiles"] = convert_to_datetime(datasets["Profiles"], "joined")
datasets["Profiles"] = convert_to_datetime(datasets["Profiles"], "birthday")

datasets["Details"] = convert_to_datetime(datasets["Details"], "start_date")
datasets["Details"] = convert_to_datetime(datasets["Details"], "end_date")

Column 'birthday' successfully converted.
Column 'joined' successfully converted.
Column 'birthday' successfully converted.
Column 'start_date' successfully converted.
Column 'end_date' successfully converted.


c:\Users\ASUS\Desktop\SaraZunino\solution\functions\utils.py:62: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  # converte in ToDateTime


Successivamente, convertiamo le restanti colonne numeriche nel tipo più adatto per l'analisi statistica. Nello specifico:
- In `Details`: Le colonne 'years', 'scored_by', 'rank' ed 'episodes' vengono convertite da float a Int64. L'uso di Int64 ci permette di trattare queste colonne come interi pur mantenendo eventuali valori nulli (NaN) senza generare errori.
- In `Profiles`: Le colonne relative allo stato di visione ('watching', 'completed', ecc.) venivano lette come object a causa della virgola usata come separatore delle migliaia. Abbiamo rimosso la virgola per permettere la corretta conversione in Int64.
- In `Characters`: La colonna 'favorites' viene uniformata al tipo Int64.

In [37]:
datasets["Details"]["year"] = datasets["Details"]["year"].astype("Int64")
datasets["Details"]["scored_by"] = datasets["Details"]["scored_by"].astype("Int64")
datasets["Details"]["rank"] = datasets["Details"]["rank"].astype("Int64")
datasets["Details"]["episodes"] = datasets["Details"]["episodes"].astype("Int64")

datasets["Characters"]["favorites"] = datasets["Characters"]["favorites"].astype("Int64")

cols_to_fix = ['watching', 'completed', 'on_hold', 'dropped', 'plan_to_watch']

for col in cols_to_fix:
    if col in datasets["Profiles"].columns:
        datasets["Profiles"][col] = pd.to_numeric(
            datasets["Profiles"][col].astype(str).str.replace(',', ''), 
            errors='coerce'
        )
        datasets["Profiles"][col] = datasets["Profiles"][col].astype("Int64")

Per concludere questa fase, visualizziamo i nuovi tipi di dato. Questo confronto finale ci permette di confermare che tutte le conversioni siano andate a buon fine.

In [38]:
for name, df in datasets.items():
    print(f"\nDATASET: {name}")
    print(df.dtypes)


DATASET: Characters
character_mal_id    float64
url                  object
name                 object
name_kanji           object
image                object
favorites             Int64
about                object
dtype: object

DATASET: Character_anime_works
anime_mal_id         int64
character_mal_id     int64
character_name      object
role                object
dtype: object

DATASET: Character_nicknames
character_mal_id     int64
nickname            object
dtype: object

DATASET: Details
mal_id                           int64
title                           object
title_japanese                  object
url                             object
image_url                       object
type                            object
status                          object
score                          float64
scored_by                        Int64
start_date         datetime64[ns, UTC]
end_date           datetime64[ns, UTC]
synopsis                        object
rank                           

## 5. Analisi e Gestione dei Valori Nulli (NaN)

In questa fase finale della pulizia, ci occupiamo dei valori mancanti all'interno dei nostri dataset.  
Utilizziamo la funzione `check_missing` per visualizzare la percentuale dei valori NaN e identificare in quali colonne specifiche si trovano.

In [43]:
for name, df in datasets.items():
    check_missing(df, name)
    print()

In Characters                col: name_kanji                missing: 26.42%
In Characters                col: about                     missing: 46.19%

In Character_anime_works     No missing values found.

In Character_nicknames       col: nickname                  missing: 0.04%

In Details                   col: title_japanese            missing: 0.41%
In Details                   col: type                      missing: 0.23%
In Details                   col: score                     missing: 34.79%
In Details                   col: scored_by                 missing: 34.79%
In Details                   col: start_date                missing: 2.94%
In Details                   col: end_date                  missing: 61.09%
In Details                   col: synopsis                  missing: 17.71%
In Details                   col: rank                      missing: 24.03%
In Details                   col: rating                    missing: 2.07%
In Details                   col: ep

Attraverso la funzione `drop_empty_rows`, eliminiamo le righe che contengono esclusivamente valori NaN. Queste righe risultano totalmente inutili alla nostra analisi.

In [40]:
for name, df in datasets.items():
    datasets[name] = drop_empty_rows(df, name)

In Characters: removed 2 empty rows


Per quanto riguarda la gestione dei restanti valori mancanti, adottiamo una strategia differente in base al tipo di dato:

- Dati Numerici (int/float): Manteniamo i valori NaN per evitare di alterare i risultati futuri.

- Dati Testuali (Object): Utilizziamo la funzione `fill_object_nan` per inserire la stringa 'Unknown'.

In [15]:
for name in datasets:
    datasets[name] = fill_object_nan(datasets[name])

Dopo aver rimosso le righe vuote e gestito i valori nulli, la colonna 'characters_mal_id' risulta univoca. Procediamo quindi alla sua conversione definitiva nel tipo int.

In [41]:
datasets["Characters"]['character_mal_id'] = datasets["Characters"]['character_mal_id'].astype(int)

C:\Users\ASUS\AppData\Local\Temp\ipykernel_11500\2218145571.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  datasets["Characters"]['character_mal_id'] = datasets["Characters"]['character_mal_id'].astype(int)


## 6. Salvataggio dei Dataset Puliti

In quest'ultima fase della preparazione, procediamo al salvataggio dei dataset elaborati. Dopo aver rimosso i duplicati, normalizzato i tipi di dato e gestito i valori mancanti, esportiamo i file in una cartella dedicata `data/processed/`.

In [22]:
for name, df in datasets.items():
    path = f"../data/processed/{name}_cleaned.csv"
    df.to_csv(path, index=False)
    print(f"Salvato: {path}")

Salvato: ../data/processed/Characters_cleaned.csv
Salvato: ../data/processed/Character_anime_works_cleaned.csv
Salvato: ../data/processed/Character_nicknames_cleaned.csv
Salvato: ../data/processed/Details_cleaned.csv
Salvato: ../data/processed/Favs_cleaned.csv
Salvato: ../data/processed/Person_alternate_names_cleaned.csv
Salvato: ../data/processed/Person_anime_works_cleaned.csv
Salvato: ../data/processed/Person_voice_works_cleaned.csv
Salvato: ../data/processed/Person_details_cleaned.csv
Salvato: ../data/processed/Profiles_cleaned.csv
Salvato: ../data/processed/Ratings_cleaned.csv
Salvato: ../data/processed/Recommendations_cleaned.csv
Salvato: ../data/processed/Stats_cleaned.csv
